# LD7187 — Fraud Detection Pipeline
**Run it in Google Colab (on GCP)**

This notebook runs the project in Colab, which is hosted on Google Cloud.

### Quick steps
1. Run the cells from top to bottom
2. Upload `creditcard.csv` when Cell 3 asks for it
3. Check outputs in `/content/outputs/`

## Cell 1 — Install packages

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost shap -q

## Cell 2 — Clone your GitHub repo

In [ ]:
# Put your real GitHub repo URL here
GITHUB_REPO = "https://github.com/YOUR_USERNAME/fraud-detection.git"

!git clone {GITHUB_REPO}

import os
# Move into the repo folder we just cloned
repo_name = GITHUB_REPO.split('/')[-1].replace('.git', '')
os.chdir(f'/content/{repo_name}')
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

## Cell 3 — Upload the dataset
Get `creditcard.csv` from https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
Then upload it using the picker below.

In [ ]:
from google.colab import files

print("Please upload creditcard.csv ...")
uploaded = files.upload()

# Move uploaded files into the current project folder
import shutil
for filename in uploaded.keys():
    shutil.move(f'/content/{filename}', f'./{filename}')
    print(f"Uploaded: {filename}")

## Cell 4 — Run the full pipeline

In [ ]:
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
warnings.filterwarnings('ignore')
np.random.seed(42)

from data_loader import load_data
from eda import run_eda
from preprocessing import preprocess
from models import train_all_models
from evaluation import evaluate_all

# Run the full pipeline
df = load_data()
run_eda(df)
X_train, y_train, X_test, y_test, feature_names = preprocess(df)
fraud_ratio = (df['Class'] == 0).sum() / (df['Class'] == 1).sum()
results = train_all_models(X_train, y_train, fraud_ratio)
metrics_df, thresh_df = evaluate_all(results, X_test, y_test)

print('\nPipeline complete!')

## Cell 5 — Show output plots

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

output_files = sorted([
    f for f in os.listdir('outputs') if f.endswith('.png')
])

print(f"Total plots generated: {len(output_files)}")

for fname in output_files:
    img = mpimg.imread(f'outputs/{fname}')
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(fname, fontsize=10)
    plt.tight_layout()
    plt.show()

## Cell 6 — Print final metrics

In [ ]:
print('=== Final Model Metrics ===')
print(metrics_df.to_string(index=False))
print('\n=== Threshold Tuning ===')
print(thresh_df.to_string(index=False))

## Cell 7 — Download all output plots

In [ ]:
import shutil
from google.colab import files

# Zip the outputs folder, then download it
shutil.make_archive('/content/fraud_detection_outputs', 'zip', 'outputs')
files.download('/content/fraud_detection_outputs.zip')
print('Downloading outputs.zip ...')